In [1]:
%pip install torch_geometric
%pip install sentence_transformers
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import random
import ssl

URI = "neo4j://be29599c.databases.neo4j.io"
AUTH = ("be29599c", "Vvmm49Tbvs4_zdwXjvEA2-W_-0_Baw6sA__AgZdsL3g")

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

driver = GraphDatabase.driver(
    URI,
    auth=AUTH,
    ssl_context=ssl_context  # only needed if ignoring SSL
)

encoder = SentenceTransformer("all-MiniLM-L6-v2")

experience_map = {
    "intern": 0,
    "fresher": 1,
    "junior": 2,
    "mid": 3,
    "senior": 4,
    "lead": 5,
    "unspecified": 0
}


def get_hetero_data(tx):
    data = HeteroData()
    id_maps = {}

    node_configs = {
        "Skill": "name",
        "Job": "title",
        "JobSeeker": "job_title"
    }

    for label, prop in node_configs.items():
        if label == "Skill":
            query = f"MATCH (n:{label}) RETURN elementId(n) as id, n.{prop} as text"
        else:
            query = f"MATCH (n:{label}) RETURN elementId(n) as id, n.{prop} as text, n.experience_level as experience"

        result = tx.run(query)

        texts, mapping, experiences = [], {}, []

        for i, rec in enumerate(result):
            mapping[rec['id']] = i
            text = rec['text'] if rec['text'] else "unknown"
            texts.append(str(text))

            if label != "Skill":
                exp = experience_map.get(rec['experience'], 0)
                experiences.append(exp)

        embeddings = encoder.encode(texts)
        x = torch.tensor(embeddings, dtype=torch.float)

        if label != "Skill":
            exp_tensor = torch.tensor(experiences).float().unsqueeze(1) / 5.0
            x = torch.cat([x, exp_tensor], dim=1)
            data[label].experience = torch.tensor(experiences, dtype=torch.float)

        data[label].x = x
        data[label].text = texts
        id_maps[label] = mapping

        if label == "Job":
           data[label].idx_to_elementId = {i: eid for eid, i in mapping.items()}

    relations = [
        ("Job", "REQUIRES_SKILL", "Skill"),
        ("JobSeeker", "HAS_SKILL", "Skill")
    ]

    for src, rel, dst in relations:
        query = f"MATCH (a:{src})-[r:{rel}]->(b:{dst}) RETURN elementId(a) as s, elementId(b) as d"
        result = tx.run(query)

        edges = []
        for rec in result:
            if rec['s'] in id_maps[src] and rec['d'] in id_maps[dst]:
                edges.append([id_maps[src][rec['s']], id_maps[dst][rec['d']]])

        if edges:
            data[src, rel, dst].edge_index = torch.tensor(edges).t().contiguous()

    return data

with driver.session() as session:
    hetero_graph = session.execute_read(get_hetero_data)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Move x_dict to device
for k in hetero_graph.x_dict:
    hetero_graph[k].x = hetero_graph[k].x.to('cpu')

In [5]:
for (src, rel, dst) in hetero_graph.edge_index_dict.keys():
    rev_key = (dst, rel + "_rev", src)
    if rev_key not in hetero_graph.edge_index_dict:
        hetero_graph[rev_key].edge_index = hetero_graph[src, rel, dst].edge_index.flip(0)

# =============================
# 8️⃣ ADD SELF-LOOPS
# =============================
for node_type in hetero_graph.node_types:
    N = hetero_graph[node_type].num_nodes
    hetero_graph[node_type, 'self_loop', node_type].edge_index = torch.arange(N).unsqueeze(0).repeat(2,1).to('cpu')

In [6]:
# =============================
# OPTIMIZED POSITIVE SAMPLING (SEMANTIC)
# =============================
import torch

def generate_positive_pairs(data, encoder, threshold=0.4, device='cuda'):
    # 1️⃣ Precompute embeddings for all skills
    skill_texts = data['Skill'].text  # List of all skill strings
    skill_embeddings = encoder.encode(
        skill_texts,
        convert_to_tensor=True,
        device=device,
        normalize_embeddings=True  # cosine similarity ready
    )  # shape: [num_skills, emb_dim]

    # 2️⃣ Map JobSeekers and Jobs to their skill indices
    js_skills_idx = {}
    for js, sk in data['JobSeeker','HAS_SKILL','Skill'].edge_index.t().tolist():
        js_skills_idx.setdefault(js, []).append(sk)

    job_skills_idx = {}
    for job, sk in data['Job','REQUIRES_SKILL','Skill'].edge_index.t().tolist():
        job_skills_idx.setdefault(job, []).append(sk)

    # 3️⃣ Pool skill embeddings per JS / Job (mean of skill embeddings)
    js_embs = {}
    for js, idx_list in js_skills_idx.items():
        js_embs[js] = skill_embeddings[idx_list].mean(dim=0)

    job_embs = {}
    for job, idx_list in job_skills_idx.items():
        job_embs[job] = skill_embeddings[idx_list].mean(dim=0)

    # 4️⃣ Generate positive pairs
    pos = []
    for js, js_emb in js_embs.items():
        for job, job_emb in job_embs.items():
            sim = (js_emb @ job_emb).item()  # cosine similarity
            if sim >= threshold:
                pos.append([js, job])

    # 5️⃣ Convert to edge_index tensor
    if pos:
        return torch.tensor(pos, dtype=torch.long).t()
    else:
        return torch.empty((2,0), dtype=torch.long)

In [7]:
from sentence_transformers import SentenceTransformer
import torch

# 1️⃣ Initialize encoder
device = 'cpu'
encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# 2️⃣ Call the function
pos_edge_index = generate_positive_pairs(
    data=hetero_graph,
    encoder=encoder,
    threshold=0.25,    # similarity threshold
    device=device
)

# 3️⃣ Inspect result
print(f"Positive pairs shape: {pos_edge_index.shape}")
print(pos_edge_index[:, :10])  # show first 10 pairs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Positive pairs shape: torch.Size([2, 12680])
tensor([[ 35,  35,  35,  35,  35,  49,  49,  49,  49,  49],
        [ 45,  55, 108, 125, 134,  10,  19,  20,  27,  30]])


In [8]:
# =============================
# NEGATIVE SAMPLING
# =============================

def hard_negative_sampling(data, pos_edge_index, num_samples):
    pos_set = set(map(tuple, pos_edge_index.t().tolist()))
    neg = set()

    js_nodes = list(range(data['JobSeeker'].num_nodes))
    job_nodes = list(range(data['Job'].num_nodes))

    while len(neg) < num_samples:
        js = random.choice(js_nodes)
        job = random.choice(job_nodes)

        if (js, job) not in pos_set:
            neg.add((js, job))

    return torch.tensor(list(neg)).t()

neg_edge_index = hard_negative_sampling(hetero_graph, pos_edge_index, pos_edge_index.size(1))


In [9]:
def split_edges(edge_index, test_size=0.2, val_size=0.5):
    edges = edge_index.t().numpy()
    train, temp = train_test_split(edges, test_size=test_size, random_state=42)
    val, test = train_test_split(temp, test_size=val_size, random_state=42)
    return torch.tensor(train).t(), torch.tensor(val).t(), torch.tensor(test).t()

# =============================
# RUN SAMPLING & SPLIT
# =============================
# Positive pairs
pos_train, pos_val, pos_test = split_edges(pos_edge_index)

# Negative pairs
neg_train, neg_val, neg_test = split_edges(neg_edge_index)

In [10]:
print("Positive train/val/test shapes:", pos_train.shape, pos_val.shape, pos_test.shape)
print("Negative train/val/test shapes:", neg_train.shape, neg_val.shape, neg_test.shape)
print("First 10 positive pairs:\n", pos_train[:, :10])
print("First 10 negative pairs:\n", neg_train[:, :10])

Positive train/val/test shapes: torch.Size([2, 10144]) torch.Size([2, 1268]) torch.Size([2, 1268])
Negative train/val/test shapes: torch.Size([2, 10144]) torch.Size([2, 1268]) torch.Size([2, 1268])
First 10 positive pairs:
 tensor([[ 889, 1008,  242, 1319, 1112,  130,  146, 1341,  898, 1235],
        [  10,   10,   84,   31,  112,   56,   23,   36,   49,   84]])
First 10 negative pairs:
 tensor([[ 128, 1222, 1332, 1298, 1034, 1259,  897, 1195,  273,  787],
        [  42,   37,  134,   14,   58,  121,   37,  130,   18,  102]])


In [11]:
class HGTModel(torch.nn.Module):
    def __init__(self, metadata, in_channels_dict, hidden=128, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.convs = torch.nn.ModuleList([
            HGTConv(in_channels_dict, hidden, metadata, heads=2),
            HGTConv(hidden, hidden, metadata, heads=2)
        ])
    def forward(self, x_dict, edge_index_dict):
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {
                k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
                for k, v in x_dict.items()
            }
        return x_dict

class Predictor(torch.nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(dim*2, 128),  # input dim = dim*2
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(128,1)
        )

    def forward(self, x1, x2):
        return self.mlp(torch.cat([x1,x2], dim=1)).squeeze()

# In channels per node type
in_channels_dict = {k: hetero_graph[k].x.shape[1] for k in hetero_graph.node_types}
model = HGTModel(hetero_graph.metadata(), in_channels_dict).to(device)
# Run one forward pass to get embedding size
with torch.no_grad():
    out = model(hetero_graph.x_dict, hetero_graph.edge_index_dict)

embed_dim = out['JobSeeker'].shape[1]

predictor = Predictor(embed_dim).to(device)
user_proj = torch.nn.Linear(384 + 1, embed_dim).to(device)

# Recreate optimizer
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(predictor.parameters()),
    lr=0.001
)

# =============================
# 13️⃣ TRAIN LOOP
# =============================
epochs = 20
batch_size = 512  # mini-batch to avoid memory issues

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    out = model(hetero_graph.x_dict, hetero_graph.edge_index_dict)
    js_emb, job_emb = out['JobSeeker'], out['Job']

    perm = torch.randperm(pos_train.size(1))
    losses = []

    total_loss = 0.0
    for i in range(0, pos_train.size(1), batch_size):
        idx = perm[i:i+batch_size]
        pos_score = predictor(js_emb[pos_train[0, idx]], job_emb[pos_train[1, idx]])
        neg_score = predictor(js_emb[neg_train[0, idx]], job_emb[neg_train[1, idx]])
        batch_loss = -torch.log(torch.sigmoid(pos_score - neg_score)).mean()
        total_loss += batch_loss

        losses.append(batch_loss.item())

    # Backward once for the total loss
    total_loss.backward()
    optimizer.step()
    print(f"Epoch {epoch} Loss: {np.mean(losses):.4f}")

Epoch 0 Loss: 0.6930
Epoch 1 Loss: 0.6931
Epoch 2 Loss: 0.6929
Epoch 3 Loss: 0.6928
Epoch 4 Loss: 0.6926
Epoch 5 Loss: 0.6924
Epoch 6 Loss: 0.6920
Epoch 7 Loss: 0.6913
Epoch 8 Loss: 0.6901
Epoch 9 Loss: 0.6890
Epoch 10 Loss: 0.6867
Epoch 11 Loss: 0.6840
Epoch 12 Loss: 0.6810
Epoch 13 Loss: 0.6752
Epoch 14 Loss: 0.6696
Epoch 15 Loss: 0.6621
Epoch 16 Loss: 0.6512
Epoch 17 Loss: 0.6398
Epoch 18 Loss: 0.6256
Epoch 19 Loss: 0.6104


In [12]:
from sklearn.metrics import roc_auc_score

def evaluate(model, predictor, graph, pos_edges, neg_edges, device='cpu'):
    model.eval()
    predictor.eval()

    with torch.no_grad():
        out = model(graph.x_dict, graph.edge_index_dict)
        js_emb = out['JobSeeker']
        job_emb = out['Job']

        # Positive scores
        pos_scores = predictor(
            js_emb[pos_edges[0]],
            job_emb[pos_edges[1]]
        ).cpu()

        # Negative scores
        neg_scores = predictor(
            js_emb[neg_edges[0]],
            job_emb[neg_edges[1]]
        ).cpu()

        # Labels
        y_true = torch.cat([
            torch.ones(pos_scores.size(0)),
            torch.zeros(neg_scores.size(0))
        ])

        y_scores = torch.cat([pos_scores, neg_scores])

        auc = roc_auc_score(y_true.numpy(), y_scores.numpy())

        return auc

In [13]:
print(out['JobSeeker'].shape)

torch.Size([1414, 128])


In [14]:
val_auc = evaluate(model, predictor, hetero_graph, pos_val, neg_val)
test_auc = evaluate(model, predictor, hetero_graph, pos_test, neg_test)

print(f"Validation AUC: {val_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

Validation AUC: 0.7895
Test AUC: 0.7800


In [15]:
def hit_at_k(model, predictor, graph, pos_edges, k=5, device='cpu'):
    model.eval()
    predictor.eval()

    hits = 0
    total = pos_edges.size(1)

    with torch.no_grad():
        # Forward pass
        out = model(
            {k: v.to(device) for k, v in graph.x_dict.items()},
            {k: v.to(device) for k, v in graph.edge_index_dict.items()}
        )

        js_emb = out['JobSeeker']
        job_emb = out['Job']

        for i in range(total):
            js = pos_edges[0, i].item()
            true_job = pos_edges[1, i].item()

            # Repeat JS embedding for all jobs
            js_vec = js_emb[js].unsqueeze(0).repeat(job_emb.size(0), 1)

            # Score all jobs
            scores = predictor(js_vec, job_emb)

            # Top-K jobs
            top_k_jobs = scores.topk(k).indices.tolist()

            if true_job in top_k_jobs:
                hits += 1

    return hits / total

In [16]:
hit5 = hit_at_k(model, predictor, hetero_graph, pos_test, k=5, device=device)
hit10 = hit_at_k(model, predictor, hetero_graph, pos_test, k=10, device=device)

print(f"Hit@5: {hit5:.4f}")
print(f"Hit@10: {hit10:.4f}")

Hit@5: 0.1585
Hit@10: 0.2539


In [17]:
def precision_at_k(model, predictor, graph, pos_edges, k=5, device='cpu'):
    model.eval()
    predictor.eval()

    precision_scores = []

    with torch.no_grad():
        out = model(
            {k: v.to(device) for k, v in graph.x_dict.items()},
            {k: v.to(device) for k, v in graph.edge_index_dict.items()}
        )

        js_emb = out['JobSeeker']
        job_emb = out['Job']

        for i in range(pos_edges.size(1)):
            js = pos_edges[0, i].item()
            true_job = pos_edges[1, i].item()

            js_vec = js_emb[js].unsqueeze(0).repeat(job_emb.size(0), 1)
            scores = predictor(js_vec, job_emb)

            top_k = scores.topk(k).indices.tolist()

            precision_scores.append(1 if true_job in top_k else 0)

    return sum(precision_scores) / len(precision_scores)

In [18]:
precision5 = precision_at_k(model, predictor, hetero_graph, pos_test, k=5, device=device)
precision10 = precision_at_k(model, predictor, hetero_graph, pos_test, k=10, device=device)

print(f"Precision@5: {precision5:.4f}")
print(f"Precision@10: {precision10:.4f}")

Precision@5: 0.1585
Precision@10: 0.2539


In [19]:
def fetch_job_skills():
    query = """
    MATCH (j:Job)-[:REQUIRES_SKILL]->(s:Skill)
    RETURN elementId(j) AS job_id, collect(s.name) AS skills
    """
    with driver.session() as session:
        result = session.run(query)
        job_skills_map = {rec['job_id']: rec['skills'] for rec in result}
    return job_skills_map

job_skills_map = fetch_job_skills()

In [20]:
print(job_skills_map)

{'4:b90b36c0-a59a-4c26-bc2a-e296cb338744:0': ['SQL', 'Kafka', 'NumPy', 'Python', 'Airflow', 'Scikit-learn', 'PyTorch', 'Pandas', 'Spark', 'TensorFlow'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:11': ['Python', 'Process Automation', 'Blue Prism', 'Automation Anywhere', 'UiPath'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:17': ['Next.js', 'Vue.js', 'Angular', 'CSS', 'Webpack', 'HTML', 'React', 'TypeScript', 'JavaScript', 'Redux'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:26': ['SAP', 'SAP HANA', 'ERP', 'ABAP', 'Business Processes'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:32': ['Unreal Engine', 'DirectX', '3D Graphics', 'OpenGL', 'Game Physics', 'C++', 'C#', 'Unity'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:38': ['Backlog Grooming', 'Agile', 'Scrum', 'Product Roadmap', 'JIRA', 'Stakeholder Management'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:45': ['Python', 'Process Automation', 'Blue Prism', 'Automation Anywhere', 'UiPath'], '4:b90b36c0-a59a-4c26-bc2a-e296cb338744:47': ['Jenkins', 'Linux',

In [21]:
def recommend_jobs(user_skills, user_experience, model, predictor, graph, encoder, top_k=5, device='cpu'):
    """
    Recommend jobs for a new user based on skills and experience using HGT + predictor.

    Args:
        user_skills (list[str]): List of user's skills (lowercase recommended).
        user_experience (str): Experience level ("intern", "junior", "mid", "senior", "lead").
        model (HGTModel): Trained HGT model.
        predictor (Predictor): Trained predictor.
        graph (HeteroData): Heterogeneous graph containing Job, Skill, JobSeeker nodes.
        encoder (SentenceTransformer): Encoder to embed skill/job texts.
        top_k (int): Number of top job recommendations to return.
        device (str): 'cpu' or 'cuda'.

    Returns:
        list[dict]: List of recommended jobs with contributing skills.
    """
    model.eval()
    predictor.eval()

    # 1️⃣ Prepare user skill embeddings
    user_skills_lower = [s.lower() for s in user_skills]
    skill_texts = [s.lower() for s in graph['Skill'].text]
    skill_indices = [i for i, s in enumerate(skill_texts) if s in user_skills_lower]

    if not skill_indices:
        print("Warning: None of the user skills match existing skills in the graph.")
        return []

    # Encode user as a vector with experience appended
    exp_value = experience_map.get(user_experience.lower(), 0) / 5.0
    skill_embs = graph['Skill'].x[skill_indices]  # shape: [num_user_skills, skill_dim]
    user_emb = skill_embs.mean(dim=0, keepdim=True)
    user_emb = torch.cat([user_emb, torch.tensor([[exp_value]], dtype=torch.float)], dim=1).to(device)

    # 2️⃣ Temporarily add a new JobSeeker node
    original_js_x = graph['JobSeeker'].x.clone()
    new_js_idx = graph['JobSeeker'].num_nodes
    graph['JobSeeker'].x = torch.cat([graph['JobSeeker'].x, user_emb], dim=0)

    # 3️⃣ Temporarily add HAS_SKILL edges for the new user
    if ('JobSeeker', 'HAS_SKILL', 'Skill') in graph.edge_index_dict:
        orig_edges = graph['JobSeeker', 'HAS_SKILL', 'Skill'].edge_index
    else:
        orig_edges = torch.empty((2,0), dtype=torch.long)

    new_edges = torch.tensor([[new_js_idx]*len(skill_indices), skill_indices], dtype=torch.long)
    graph['JobSeeker', 'HAS_SKILL', 'Skill'].edge_index = torch.cat([orig_edges, new_edges], dim=1)

    # 4️⃣ Forward pass through HGT
    with torch.no_grad():
        out = model({k: v.to(device) for k,v in graph.x_dict.items()},
                    {k: v.to(device) for k,v in graph.edge_index_dict.items()})

    js_emb = out['JobSeeker']
    job_emb = out['Job']

    # 5️⃣ Score all jobs
    user_vec = js_emb[new_js_idx].unsqueeze(0).repeat(job_emb.size(0), 1)
    scores = predictor(user_vec, job_emb).cpu()

    # 6️⃣ Rank jobs and filter by contributing skills
    ranked_indices = scores.topk(top_k*25).indices.tolist()  # get more to filter later
    recommended_jobs = []

    for idx in ranked_indices:
        job_title = graph['Job'].text[idx]
        job_skills = [graph['Skill'].text[s] for s in graph['Job','REQUIRES_SKILL','Skill'].edge_index[1][
            (graph['Job','REQUIRES_SKILL','Skill'].edge_index[0]==idx)
        ].tolist()]

        contributing_skills = [s for s in job_skills if s.lower() in user_skills_lower]
        if not contributing_skills:
            continue  # skip jobs with no actual matching skills

        recommended_jobs.append({
            'job_title': job_title,
            'recommended_skills': job_skills,
            'contributing_skills': contributing_skills
        })

        if len(recommended_jobs) >= top_k:
            break

    # 7️⃣ Restore original graph
    graph['JobSeeker'].x = original_js_x
    graph['JobSeeker','HAS_SKILL','Skill'].edge_index = orig_edges

    return recommended_jobs

In [22]:
#user_skills = ["Pandas", "Kafka", "Scikit-learn", "TensorFlow", "Airflow"]
#user_skills = ["Selenium", "Postman", "Load Testing", "JUnit", "Automation Testing", "API Testing", "Cypress"]
#user_skills = ["Java", "REST APIs", "Data Structures", "Algorithms"]
user_skills = ["AWS", "Linux", "Networking", "Kubernetes", "IAM", "Azure", "Security Scanning"]
user_experience = "senior"

top_jobs = recommend_jobs(user_skills, user_experience, model, predictor, hetero_graph, encoder, top_k=5)
for job in top_jobs:
    print(job)

{'job_title': 'devsecops engineer', 'recommended_skills': ['Linux', 'Terraform', 'Kubernetes', 'CI/CD', 'AWS', 'Docker', 'DAST', 'Security Scanning', 'SAST'], 'contributing_skills': ['Linux', 'Kubernetes', 'AWS', 'Security Scanning']}
{'job_title': 'devsecops engineer', 'recommended_skills': ['Linux', 'Terraform', 'Kubernetes', 'CI/CD', 'AWS', 'Docker', 'DAST', 'Security Scanning', 'SAST'], 'contributing_skills': ['Linux', 'Kubernetes', 'AWS', 'Security Scanning']}
{'job_title': 'devsecops engineer', 'recommended_skills': ['Linux', 'Terraform', 'Kubernetes', 'CI/CD', 'AWS', 'Docker', 'DAST', 'Security Scanning', 'SAST'], 'contributing_skills': ['Linux', 'Kubernetes', 'AWS', 'Security Scanning']}
{'job_title': 'devsecops engineer', 'recommended_skills': ['Linux', 'Terraform', 'Kubernetes', 'CI/CD', 'AWS', 'Docker', 'DAST', 'Security Scanning', 'SAST'], 'contributing_skills': ['Linux', 'Kubernetes', 'AWS', 'Security Scanning']}
{'job_title': 'devsecops engineer', 'recommended_skills': ['

In [23]:
import os
import torch

# Create folder if it doesn't exist
os.makedirs("saved", exist_ok=True)

# Save trained weights
torch.save(model.state_dict(), "saved/model.pt")
torch.save(predictor.state_dict(), "saved/predictor.pt")

print("✅ Model and predictor saved successfully.")

✅ Model and predictor saved successfully.
